# Render Model Fitting: Group-Level Models

Fit one parameter vector across all subjects for the selected manuscript-scale models.


In [1]:
import os
import papermill as pm
from lpp_ecmr.fitting_config import BASE_PARAMS, ECMR_BASE_PARAMS
from lpp_ecmr.full_ecmr_registry import ECMR_MODELS
from lpp_ecmr.single_context_registry import SINGLE_CONTEXT_MODELS

_here = os.path.dirname(os.path.abspath("__file__"))
_lpp_ecmr = os.path.dirname(_here)
_workspace = os.path.dirname(_lpp_ecmr)

template = os.path.join(_workspace, "jaxcmr", "templates", "fitting.ipynb")
rendered_dir = os.path.join(_here, "rendered")
os.makedirs(rendered_dir, exist_ok=True)

In [2]:
# Build manuscript model configs
manuscript_models = []

for m in SINGLE_CONTEXT_MODELS:
    if m["model_name"] in ("WeirdCMRNoStop", "EEGLPPParsimonious"):
        manuscript_models.append(BASE_PARAMS.copy() | {k: v for k, v in m.items() if k != "enabled"})

for m in ECMR_MODELS:
    if m["model_name"] in ("eCMREmotionOnly", "eCMREmotionTimesLPP"):
        manuscript_models.append(ECMR_BASE_PARAMS.copy() | {k: v for k, v in m.items() if k != "enabled"})

print(f"{len(manuscript_models)} models to fit")
for m in manuscript_models:
    print(f"  {m['model_name']}: {len(m['parameters']['free'])} free params")

4 models to fit
  WeirdCMRNoStop: 9 free params
  EEGLPPParsimonious: 10 free params
  eCMREmotionOnly: 10 free params
  eCMREmotionTimesLPP: 11 free params


In [ ]:
for model_cfg in manuscript_models:
    model_name = model_cfg["model_name"]

    params = model_cfg.copy()
    params["fit_to_subjects"] = False
    params["redo_fits"] = False
    params["redo_sims"] = False
    params["base_run_tag"] = "group_level"
    params["target_directory"] = ""
    params["figure_dir"] = "figures/fitting"
    params["display_iterations"] = True
    params["figure_str"] = f"TalmiEEG_{model_name}_group_level_best_of_3"

    output = os.path.join(rendered_dir, f"group_fitting_{model_name}.ipynb")
    print(output)

    pm.execute_notebook(
        template,
        output,
        parameters=params,
        cwd=_lpp_ecmr,
        progress_bar=True,
    )